# Text Preprocessing & Embeddings for LLMs

**Based on Chapter 2 of *Build a Large Language Model (From Scratch)* by Sebastian Raschka**

This notebook walks through the full text-preprocessing pipeline that feeds a GPT-style large language model:

1. **Tokenization** — converting raw text into integer token IDs using byte-pair encoding (BPE).
2. **Sliding-window dataset creation** — building input / target pairs for next-token prediction.
3. **Token & positional embeddings** — mapping discrete IDs into continuous vector representations the transformer can learn from.

Along the way I add my own explanations on *why* each step matters—both for LLMs and for agentic AI systems—and I run a small experiment varying `max_length` and `stride` to show how overlap affects training data.

---
## 1. Import Libraries & Load Text Data

In [1]:
import re
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("tiktoken version:", tiktoken.__version__)

PyTorch version: 2.10.0+cpu
tiktoken version: 0.12.0


In [2]:
# Load the raw text (The Verdict by Edith Wharton — public domain)
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text)}")
print(f"Preview:\n{raw_text[:200]}")

Total characters: 20479
Preview:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


---
## 2. Tokenization with tiktoken (BPE)

In [3]:
# Instantiate GPT-2's BPE tokenizer via tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

# Encode the full text
enc_text = tokenizer.encode(raw_text)
print(f"Total tokens: {len(enc_text)}")
print(f"First 20 token IDs: {enc_text[:20]}")

Total tokens: 5145
First 20 token IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


In [4]:
# Round-trip sanity check: encode → decode
sample_text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
sample_ids = tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})
print("Token IDs:", sample_ids)
print("Decoded  :", tokenizer.decode(sample_ids))

Token IDs: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Decoded  : Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


---
## 💡 Explanation 1 — Why Tokenization Matters for LLMs & Agentic Systems

Tokenization is the **absolute first gate** between the human world and the neural-network world.  Neural networks only understand numbers, so raw text *must* be broken into discrete integer IDs before anything else can happen.

**Why BPE in particular?**

| Property | Benefit |
|----------|---------|
| Sub-word granularity | Rare or never-seen words (e.g. *"unfamiliarword"*) are split into known sub-units like `["unfam", "iliar", "word"]`, so the model never hits a dead-end `<UNK>` token. |
| Compact vocabulary | BPE keeps the vocab manageable (~50 k for GPT-2) instead of exploding to millions of word-level entries. A smaller vocab means a smaller embedding matrix and faster softmax. |
| Language agnostic | Because BPE works at the byte level it can handle any Unicode script, code, or emoji—critical for agentic systems that receive arbitrary, unpredictable user inputs. |

For **agentic systems** this robustness is non-negotiable: an agent might need to parse a user's misspelled query, a JSON tool response, or a foreign-language snippet—all in the same turn.  BPE handles all of these gracefully without requiring a hand-curated dictionary.

---
## 3. Building a Token Vocabulary & Encode/Decode Round-Trip

In [5]:
# Quick illustration: build a simple vocabulary from The Verdict (as in the book)
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: idx for idx, token in enumerate(all_tokens)}

print(f"Simple vocabulary size: {len(vocab)}")
print("Last 5 entries:", dict(list(vocab.items())[-5:]))

Simple vocabulary size: 1132
Last 5 entries: {'younger': 1127, 'your': 1128, 'yourself': 1129, '<|endoftext|>': 1130, '<|unk|>': 1131}


In [6]:
# Demonstrate encode/decode with tiktoken (the real tokenizer we'll use)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print("Original :", text)
print("Token IDs:", ids)
print("Decoded  :", decoded)
print("Match?   :", text == decoded)

Original : "It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.
Token IDs: [1, 1026, 338, 262, 938, 339, 13055, 11, 345, 760, 553, 9074, 13, 402, 271, 10899, 531, 351, 27322, 540, 11293, 13]
Decoded  : "It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.
Match?   : True


---
## 💡 Explanation 2 — Why Vocabulary Construction & Integer Encoding Matter

Once we have a vocabulary, every token maps to a **unique integer**.  This integer is the *index* into the embedding matrix, which converts the sparse, discrete world of text into the **dense, continuous vector space** that neural networks actually compute over.

Key points:

* **Vocabulary size → embedding matrix rows.**  GPT-2's BPE vocabulary has 50 257 entries, so its embedding weight matrix is `(50257, d_model)`.  A bigger vocab captures more fine-grained distinctions but costs more memory and compute; a smaller vocab forces the model to reconstruct meaning from sub-word pieces.

* **Integer encoding is lossless** (within the vocabulary).  The encode → decode round-trip reproduces the original text exactly—no information is thrown away at this stage.

* **Generalization across tasks.**  Because the same vocabulary and embedding matrix are reused for every prompt, the model amortizes its learned representations across all downstream tasks.  In an **agentic pipeline** (e.g., ReAct, tool-use chains) the same tokenizer and embeddings handle the user query, the system prompt, tool descriptions, and tool outputs—uniformity here is what makes multi-step reasoning tractable.

---
## 4. Sliding-Window Dataset & DataLoader

In [7]:
class GPTDatasetV1(Dataset):
    """Sliding-window dataset for next-token prediction."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, \
            "Token sequence must be longer than max_length"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last,
                      num_workers=num_workers)

In [8]:
# Baseline: max_length=4, stride=4 (no overlap) — as in the book
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nBatch shape:", inputs.shape)  # (8, 4)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Batch shape: torch.Size([8, 4])


---
## 💡 Explanation 3 — Why Chunking Text into Overlapping Sequences Matters

LLMs are trained with a **next-token prediction** objective: given the previous *n* tokens (the *context window*), predict token *n + 1*.  To create training examples we slide a fixed-size window across the full token sequence.

Two parameters control the window:

| Parameter | Role |
|-----------|------|
| `max_length` | Size of each context window (= number of input tokens per sample). |
| `stride` | How many tokens the window advances between consecutive samples. |

When **stride = max_length** the windows are non-overlapping—every token appears in exactly one training sample.  When **stride < max_length** the windows overlap—each token appears in *multiple* samples, each time with a different surrounding context.

**Why overlap is useful:**

1. **More training examples** from the same corpus → the model sees more gradient updates per epoch.
2. **Context-boundary diversity** — a token that was near the *end* of one window appears near the *middle* of another, so the model learns to use both short and long preceding contexts.
3. **Smoother gradients** — overlapping batches act like a form of data augmentation, reducing variance in gradient estimates.

For **agentic systems** this translates to a model that generalizes better to the varied, unpredictable context lengths an agent encounters at inference time (short tool outputs, long user instructions, etc.).

---
## 5. 🧪 Experiment — Varying `max_length` and `stride`

In [9]:
total_tokens = len(enc_text)

configs = [
    # (max_length, stride, label)
    (4,   4,   "4 / 4   (no overlap)"),
    (4,   2,   "4 / 2   (50% overlap)"),
    (4,   1,   "4 / 1   (75% overlap)"),
    (8,   8,   "8 / 8   (no overlap)"),
    (8,   4,   "8 / 4   (50% overlap)"),
    (16,  16,  "16 / 16 (no overlap)"),
    (16,  8,   "16 / 8  (50% overlap)"),
    (64,  64,  "64 / 64 (no overlap)"),
    (64,  32,  "64 / 32 (50% overlap)"),
    (256, 256, "256/256 (no overlap)"),
    (256, 128, "256/128 (50% overlap)"),
]

rows = []
for ml, st, label in configs:
    ds = GPTDatasetV1(raw_text, tokenizer, max_length=ml, stride=st)
    expected = (total_tokens - ml) // st   # formula
    rows.append({
        "max_length": ml,
        "stride": st,
        "samples": len(ds),
        "expected (formula)": expected,
        "label": label,
    })

# Print as a formatted table
print(f"Total tokens in corpus: {total_tokens}\n")
print(f"{'Label':<28} {'max_len':>7} {'stride':>6} {'samples':>8} {'expected':>8}")
print("-" * 62)
for r in rows:
    print(f"{r['label']:<28} {r['max_length']:>7} {r['stride']:>6} {r['samples']:>8} {r['expected (formula)']:>8}")

Total tokens in corpus: 5145

Label                        max_len stride  samples expected
--------------------------------------------------------------
4 / 4   (no overlap)               4      4     1286     1285
4 / 2   (50% overlap)              4      2     2571     2570
4 / 1   (75% overlap)              4      1     5141     5141
8 / 8   (no overlap)               8      8      643      642
8 / 4   (50% overlap)              8      4     1285     1284
16 / 16 (no overlap)              16     16      321      320
16 / 8  (50% overlap)             16      8      642      641
64 / 64 (no overlap)              64     64       80       79
64 / 32 (50% overlap)             64     32      159      158
256/256 (no overlap)             256    256       20       19
256/128 (50% overlap)            256    128       39       38


In [10]:
# Visual: show overlapping tokens for max_length=4, stride=2
print("=== Overlapping windows (max_length=4, stride=2) ===\n")
ds_overlap = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=2)
for i in range(min(6, len(ds_overlap))):
    inp, tgt = ds_overlap[i]
    print(f"Sample {i}: tokens {inp.tolist()}  →  \"{tokenizer.decode(inp.tolist())}\"")
print("\nNotice how consecutive samples share tokens — that's the overlap.")

=== Overlapping windows (max_length=4, stride=2) ===

Sample 0: tokens [40, 367, 2885, 1464]  →  "I HAD always"
Sample 1: tokens [2885, 1464, 1807, 3619]  →  "AD always thought Jack"
Sample 2: tokens [1807, 3619, 402, 271]  →  " thought Jack Gis"
Sample 3: tokens [402, 271, 10899, 2138]  →  " Gisburn rather"
Sample 4: tokens [10899, 2138, 257, 7026]  →  "burn rather a cheap"
Sample 5: tokens [257, 7026, 15632, 438]  →  " a cheap genius--"

Notice how consecutive samples share tokens — that's the overlap.


### Experiment Results — Analysis

The table above confirms the relationship:

$$\text{num\_samples} = \left\lfloor \frac{N - \text{max\_length}}{\text{stride}} \right\rfloor$$

where $N$ is the total number of tokens in the corpus.

**Key observations:**

| Observation | Detail |
|-------------|--------|
| **Halving the stride ≈ doubles the samples** | Going from `stride=max_length` to `stride=max_length/2` nearly doubles the dataset size every time. |
| **stride = 1 is the extreme case** | With `max_length=4, stride=1` we get ~5 k samples from ~5 k tokens—almost one sample per token—but each pair shares 3 of 4 tokens with its neighbor. |
| **Larger `max_length` → fewer samples** | A 256-token window produces far fewer (but longer) samples; overlap helps compensate. |
| **Diminishing returns of heavy overlap** | Very small strides create near-duplicate samples, which can slow convergence or cause overfitting if not paired with proper shuffling. |

**Why overlap is useful in practice:** it acts as *implicit data augmentation*—each token is seen in multiple surrounding contexts, giving the model a richer signal about how tokens relate to each other.  This leads to smoother loss landscapes and better generalization, especially on small corpora like this short story.

---
## 6. Token Embeddings & Positional Embeddings

In [11]:
# --- Small illustrative example (as in the book) ---
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size_small = 6
output_dim_small = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size_small, output_dim_small)
print("Embedding weight matrix:\n", embedding_layer.weight)
print("\nLookup for ids [2, 3, 5, 1]:\n", embedding_layer(input_ids))

Embedding weight matrix:
 Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Lookup for ids [2, 3, 5, 1]:
 tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [12]:
# --- Full-scale: GPT-2 sized embeddings ---
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer   = torch.nn.Embedding(max_length, output_dim)

# Get a batch from the dataloader
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Input token IDs:\n", inputs)
print("Shape:", inputs.shape)   # (8, 4)

Input token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Shape: torch.Size([8, 4])


In [13]:
# Token embeddings
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)  # (8, 4, 256)

# Positional embeddings
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("Positional embeddings shape:", pos_embeddings.shape)  # (4, 256)

# Final input embeddings = token + positional
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)  # (8, 4, 256)

Token embeddings shape: torch.Size([8, 4, 256])
Positional embeddings shape: torch.Size([4, 256])
Final input embeddings shape: torch.Size([8, 4, 256])


---
## 💡 Explanation 4 — Why Do Embeddings Encode Meaning, and How Are They Related to NN Concepts?

This is the central conceptual question of the chapter, so let's unpack it carefully.

### Why embeddings encode meaning

The short answer is the **distributional hypothesis**: *"You shall know a word by the company it keeps"* (J. R. Firth, 1957).  During training the model's loss function (cross-entropy over next-token prediction) pushes the embedding vectors of tokens that appear in **similar contexts** closer together in the continuous vector space, and pushes unrelated tokens apart.  After billions of gradient updates, the geometry of that vector space *mirrors* semantic relationships:

* "king" − "man" + "woman" ≈ "queen"  (classic word2vec example)
* Synonyms cluster together; antonyms are far apart on certain axes.

So meaning is not hard-coded—it **emerges** from optimization.  The model never receives an explicit dictionary; it discovers semantic structure purely from co-occurrence statistics amplified by back-propagation.

### How embeddings relate to core neural-network concepts

| NN Concept | Embedding Connection |
|------------|---------------------|
| **Learnable parameters** | The embedding matrix `W ∈ ℝ^{V × d}` is a standard weight matrix updated by gradient descent, just like any other layer. |
| **Linear layer equivalence** | Looking up row *i* of `W` is mathematically identical to multiplying `W` by a one-hot vector `e_i`.  So an embedding *is* a linear layer with a sparse (one-hot) input—just implemented more efficiently. |
| **Dimensionality reduction** | One-hot vectors live in a `V`-dimensional space (50 257-D for GPT-2); embeddings compress them into a much smaller `d`-dimensional space (e.g., 256-D), discarding noise and retaining the structure the model needs. |
| **Representation learning** | Embeddings are the very first *learned representation* in the network.  Every subsequent transformer layer (self-attention, feed-forward) *refines* these vectors, but the quality of the initial embedding sets the ceiling. |
| **Positional embeddings** | Self-attention is permutation-invariant—it treats tokens as a set, not a sequence.  Adding a positional embedding restores **order information**, which is critical because language meaning is order-dependent ("dog bites man" ≠ "man bites dog"). |

### Relevance to agentic systems

* **Semantic search & RAG** — Embedding a user query and a knowledge-base chunk into the same space lets an agent *retrieve* the most relevant context via cosine similarity, drastically reducing hallucinations.
* **Tool selection** — An agent can embed tool descriptions and compare them to the current sub-goal embedding, picking the best tool without brittle string matching.
* **Grounded reasoning** — High-quality embeddings give the model a continuous, differentiable "world model" that downstream attention layers can query, enabling multi-step planning and chain-of-thought reasoning.

In short: **embeddings are the bridge between discrete symbols and the continuous mathematics of neural networks.**  Without them, no transformer, no attention mechanism, no LLM—and no agentic system—would work.

---
## Summary

| Step | What it does | Why it matters |
|------|-------------|----------------|
| **Tokenization (BPE)** | Converts raw text → integer token IDs | Makes text consumable by neural networks; handles any input robustly |
| **Sliding-window dataset** | Creates (input, target) pairs for next-token prediction | Provides the supervised signal the model trains on; overlap = data augmentation |
| **Token embeddings** | Maps each token ID → dense vector | Learns semantic meaning via gradient descent |
| **Positional embeddings** | Adds order information to each position | Restores sequence structure that attention alone cannot capture |
| **Token + Positional = Input embedding** | Final input to the transformer | The rich, continuous representation that all downstream layers refine |

All code is adapted from **Chapter 2** of *Build a Large Language Model (From Scratch)* by **Sebastian Raschka**.